# Train Classifier

This notebook trains a dice face classifier on cropped dice images.

Goals:
- load cropped training and validation datasets
- apply image augmentation
- train an EfficientNet-based classifier
- fine-tune the backbone
- save the best classifier model

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

from project_config import (
    PROJECT_ROOT,
    CLASSIFICATION_DIR,
    MODELS_DIR,
    SEED,
    CLASSIFIER_IMG_SIZE,
    CLASSIFIER_BATCH_SIZE,
    CLASSIFIER_EPOCHS,
)

In [ ]:
tf.keras.utils.set_random_seed(SEED)

TRAIN_DIR = CLASSIFICATION_DIR / "train"
VAL_DIR = CLASSIFICATION_DIR / "val"
MODEL_PATH = MODELS_DIR / "dice_classifier_best.keras"
AUTOTUNE = tf.data.AUTOTUNE

assert TRAIN_DIR.exists(), f"Missing train dir: {TRAIN_DIR}"
assert VAL_DIR.exists(), f"Missing val dir: {VAL_DIR}"

print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("MODEL_PATH:", MODEL_PATH)

In [ ]:
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    image_size=CLASSIFIER_IMG_SIZE,
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    image_size=CLASSIFIER_IMG_SIZE,
    batch_size=CLASSIFIER_BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)

print("Class names:", class_names)
assert class_names == ["1", "2", "3", "4", "5", "6"], f"Unexpected class names: {class_names}"

In [ ]:
train_counts = {i: 0 for i in range(num_classes)}
for _, labels in train_ds_raw:
    labels_np = labels.numpy()
    for i in range(num_classes):
        train_counts[i] += int((labels_np == i).sum())

total_train = sum(train_counts.values())

class_weight = {
    i: total_train / (num_classes * max(train_counts[i], 1))
    for i in range(num_classes)
}

print("Train counts:", {class_names[i]: train_counts[i] for i in train_counts})
print("Class weights:", {class_names[i]: round(class_weight[i], 4) for i in class_weight})

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.04),
    keras.layers.RandomZoom(0.06),
    keras.layers.RandomTranslation(0.03, 0.03),
    keras.layers.RandomContrast(0.10),
], name="data_augmentation")

def preprocess_train(images, labels):
    images = tf.cast(images, tf.float32)
    images = data_augmentation(images, training=True)
    images = keras.applications.efficientnet.preprocess_input(images)
    return images, labels

def preprocess_eval(images, labels):
    images = tf.cast(images, tf.float32)
    images = keras.applications.efficientnet.preprocess_input(images)
    return images, labels

train_ds = train_ds_raw.map(preprocess_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds_raw.map(preprocess_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
inputs = keras.Input(shape=(CLASSIFIER_IMG_SIZE[0], CLASSIFIER_IMG_SIZE[1], 3))

base_model = keras.applications.EfficientNetB1(
    include_top=False,
    weights="imagenet",
    input_tensor=inputs,
)

base_model.trainable = False

x = base_model.output
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.30)(x)
outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="dice_classifier")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        mode="max",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        mode="max",
        verbose=1,
    ),
]

In [ ]:
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
base_model.trainable = True

fine_tune_from = int(len(base_model.layers) * 0.7)
for i, layer in enumerate(base_model.layers):
    if i < fine_tune_from:
        layer.trainable = False
    else:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=len(history_stage1.history["loss"]),
    epochs=CLASSIFIER_EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

print("Best model saved to:", MODEL_PATH)